In [1]:
import pandas as pd
import numpy as np
from decimal import Decimal
import glob
import backtrader as bt
import matplotlib
import matplotlib.pyplot as plt
import pandas_ta as ta
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Ensure plots are rendered inline in the notebook
pio.renderers.default = "notebook"
matplotlib.use('Agg')
%matplotlib inline

In [47]:
symbol = "USDJPY"
year = 2008
end_year = 2018
decimals = 2
multiplier = 100

In [33]:
# Define the path to your CSV files
file_path_pattern = f'/root/github/CTI_Scripts/py_scripts/backtesting/historical_data/{symbol}/DAT_MT_{symbol}_M1_*.csv'

# Define the column names
column_names = ['DATE', 'TIME', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']

# Load and combine all CSV files
all_files = glob.glob(file_path_pattern)
data_list = []

for file in all_files:
    print(f"Loading file: {file}")  # Debugging: Print file being loaded
    df = pd.read_csv(file, delimiter=',', names=column_names, header=None, dtype=str)
    # print(df.head())  # Debugging: Print the first few rows of the loaded DataFrame
    data_list.append(df)

combined_data = pd.concat(data_list)

# Combine <DATE> and <TIME> into a single datetime column
combined_data['datetime'] = pd.to_datetime(
    combined_data['DATE'] + ' ' + combined_data['TIME'], format='%Y.%m.%d %H:%M', errors='coerce'
)

# Drop rows with NaT in datetime column
combined_data.dropna(subset=['datetime'], inplace=True)

# Set datetime as the index
combined_data.set_index('datetime', inplace=True)
combined_data.sort_index(inplace=True)

# Convert numeric columns to appropriate data types
numeric_columns = ['OPEN', 'HIGH', 'LOW', 'CLOSE', 'TICKVOL', 'VOL', 'SPREAD']
combined_data[numeric_columns] = combined_data[numeric_columns].apply(pd.to_numeric, errors='coerce')

# Drop rows with NaN values in numeric columns
combined_data.replace([np.inf, -np.inf], np.nan, inplace=True)
combined_data.fillna(0, inplace=True)

# Display the final combined data
print("Final combined data:")
combined_data.head()

Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_2012.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_2016.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_2005.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_2021.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_202201030000_202302201721.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_2014.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_2019.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT_USDJPY_M1_2017.csv
Loading file: /root/github/CTI_Scripts/py_scripts/backtesting/historical_data/USDJPY/DAT_MT

,DATE,TIME,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,,,
2005-01-03 01:48:00,2005.01.03,01:48,102.58,102.58,102.58,102.58,0,0.0,0.0
2005-01-03 01:53:00,2005.01.03,01:53,102.58,102.91,102.58,102.90,0,0.0,0.0
2005-01-03 01:54:00,2005.01.03,01:54,102.91,102.91,102.91,102.91,0,0.0,0.0
2005-01-03 01:55:00,2005.01.03,01:55,102.90,102.90,102.90,102.90,0,0.0,0.0
2005-01-03 01:56:00,2005.01.03,01:56,102.88,102.88,102.85,102.86,0,0.0,0.0


In [34]:
combined_data_1m = combined_data

combined_data_1m = combined_data_1m.drop(columns=['DATE', 'TIME'])

combined_data_1m.head()

,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,
2005-01-03 01:48:00,102.58,102.58,102.58,102.58,0,0.0,0.0
2005-01-03 01:53:00,102.58,102.91,102.58,102.90,0,0.0,0.0
2005-01-03 01:54:00,102.91,102.91,102.91,102.91,0,0.0,0.0
2005-01-03 01:55:00,102.90,102.90,102.90,102.90,0,0.0,0.0
2005-01-03 01:56:00,102.88,102.88,102.85,102.86,0,0.0,0.0


In [38]:
# Calculate indicators
aroon_length = 50
ema_length = 20
stoch_rsi_length = 14
stoch_rsi_smoothK = 3
stoch_rsi_smoothD = 3
supertrend_length = 50
supertrend_mult = 3.0
kc_length = 20
kc_scalar = 1.5
atr_length = 14

combined_data_1m['AROON-OSC'] = ta.aroon(
    combined_data_1m['HIGH'],
    combined_data_1m['LOW'],
    length=aroon_length
)[f'AROONOSC_{aroon_length}'].round(decimals)
combined_data_1m['EMA_50'] = ta.ema(combined_data_1m['CLOSE'], length=ema_length).round(decimals)
combined_data_1m['STOCH-RSId'] = ta.stochrsi(
    combined_data_1m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSId_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_1m['STOCH-RSIk'] = ta.stochrsi(
    combined_data_1m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSIk_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_1m['SUPERTREND'] = ta.supertrend(
    combined_data_1m['HIGH'],
    combined_data_1m['LOW'],
    combined_data_1m['CLOSE'],
    length=supertrend_length,
    multiplier=supertrend_mult
)[f'SUPERT_{supertrend_length}_{supertrend_mult}'].round(decimals)
combined_data_1m['Keltner-Upper'] = ta.kc(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCUe_{kc_length}_{kc_scalar}'].round(decimals)
combined_data_1m['Keltner-Basis'] = ta.kc(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCBe_{kc_length}_{kc_scalar}'].round(decimals)
combined_data_1m['Keltner-Lower'] = ta.kc(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCLe_{kc_length}_{kc_scalar}'].round(decimals)
combined_data_1m['ATR'] = ta.atr(
    high=combined_data_1m['HIGH'],
    low=combined_data_1m['LOW'],
    close=combined_data_1m['CLOSE'],
    length=atr_length
).round(decimals)



# Drop rows with NaN values (due to indicator calculation)
combined_data_1m.dropna(inplace=True)

# Display the data with indicators
print("Data with indicators:")
combined_data_1m.head()

Data with indicators:


,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,AROON-OSC,EMA_50,STOCH-RSId,STOCH-RSIk,SUPERTREND,Keltner-Upper,Keltner-Basis,Keltner-Lower,ATR
datetime,,,,,,,,,,,,,,,,
2005-01-03 02:46:00,102.75,102.75,102.70,102.74,0,0.0,0.0,66.0,102.79,20.49,36.39,102.62,102.84,102.79,102.74,0.03
2005-01-03 02:47:00,102.75,102.76,102.72,102.73,0,0.0,0.0,66.0,102.78,34.30,46.52,102.64,102.84,102.78,102.73,0.03
2005-01-03 02:48:00,102.74,102.79,102.74,102.77,0,0.0,0.0,-26.0,102.78,49.28,64.92,102.66,102.84,102.78,102.73,0.04
2005-01-03 02:49:00,102.76,102.79,102.76,102.76,0,0.0,0.0,-26.0,102.78,63.30,78.45,102.67,102.83,102.78,102.73,0.03
2005-01-03 02:50:00,102.77,102.77,102.75,102.77,0,0.0,0.0,-26.0,102.78,79.98,96.57,102.67,102.83,102.78,102.73,0.03


In [64]:
plotly_data = combined_data_1m.loc['2021-01-01':'2021-01-31']

# Create a Plotly figure with subplots
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.02, 
                    row_heights=[0.10, 0.75, 0.15, 0.15],
                    subplot_titles=('ATR 50',
                                    'EMA 50 | Supertrend 50, 3.0 | Keltner Channels 20, 1.5',
                                    'Aroon Oscillator 50',
                                    'Stochastic RSI 14, 3, 3'
                                )
                            )

# Add candlestick chart
fig.add_trace(go.Candlestick(x=plotly_data.index,
                             open=plotly_data['OPEN'],
                             high=plotly_data['HIGH'],
                             low=plotly_data['LOW'],
                             close=plotly_data['CLOSE'],
                             ),
            row=2, col=1)

# Add ATR to top of chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['ATR'],
    mode='lines',
    line=go.scatter.Line(color='yellow'),
    name='ATR'
), row=1, col=1)
# Add indicators to candlestick chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['EMA_50'],
    mode='lines',
    line=go.scatter.Line(color='red'),
    name='EMA 50'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['SUPERTREND'],
    mode='lines',
    line=go.scatter.Line(color='green'),
    name='SUPERTREND'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Upper'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Upper'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Basis'],
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Keltner-Basis'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Lower'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Lower'
), row=2, col=1)

# Add Aroon Oscillator
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['AROON-OSC'],
    mode='lines',
    line=go.scatter.Line(color='purple'),
    name='Aroon Oscillator'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[0] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='grey', dash='dash'),
    name='Aroon Osc Zero Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Mid Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[-60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Upper Line'
), row=3, col=1)

# Add Stochastic RSI
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSId'],
    mode='lines',
    line=go.scatter.Line(color='orange'),
    name='Stochastic RSI %D'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSIk'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Stochastic RSI %K'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[20] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='red', dash='dash'),
    name='Oversold Threshold'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[80] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='green', dash='dash'),
    name='Overbought Threshold'
), row=4, col=1)


fig.update_layout(
    title='EURUSD 1m Candlestick Chart with Indicators January 2021',
    xaxis2=dict(
        rangeslider=dict(visible=False),
        rangebreaks=[
            dict(bounds=["sat", "mon"]),  # hide weekends
            dict(values=["2021-01-08 17:00", "2021-01-10 17:00"])
        ]
    ),
    yaxis=dict(title='Price', autorange=True),
    dragmode='zoom',
    uirevision='dynamic',
)

fig.write_html('./plotly/test_plot_EURUSD_1m.html')

In [39]:
combined_data_5m = combined_data.resample('5min').agg({
    'OPEN': 'first',
    'HIGH': 'max',
    'LOW': 'min',
    'CLOSE': 'last',
    'TICKVOL': 'sum',  # Aggregate tick volumes
    'VOL': 'sum',      # Aggregate actual volumes
    'SPREAD': 'mean',  # Average spread
}).dropna()

# Ensure datetime is set as the index
combined_data_5m.reset_index(inplace=True)  # Ensure 'datetime' is a column
combined_data_5m['datetime'] = pd.to_datetime(combined_data_5m['datetime'])  # Convert to datetime
combined_data_5m.set_index('datetime', inplace=True)  # Set as index

# Display the resampled data
combined_data_5m.head()

,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD
datetime,,,,,,,
2005-01-03 01:45:00,102.58,102.58,102.58,102.58,0,0.0,0.0
2005-01-03 01:50:00,102.58,102.91,102.58,102.91,0,0.0,0.0
2005-01-03 01:55:00,102.90,102.90,102.79,102.80,0,0.0,0.0
2005-01-03 02:00:00,102.81,102.89,102.80,102.87,0,0.0,0.0
2005-01-03 02:05:00,102.88,102.91,102.88,102.90,0,0.0,0.0


In [40]:
# Calculate indicators
aroon_length = 50
ema_length = 20
stoch_rsi_length = 14
stoch_rsi_smoothK = 3
stoch_rsi_smoothD = 3
supertrend_length = 50
supertrend_mult = 3.0
kc_length = 20
kc_scalar = 1.5
atr_length = 14

combined_data_5m['AROON-OSC'] = ta.aroon(
    combined_data_5m['HIGH'],
    combined_data_5m['LOW'],
    length=aroon_length
)[f'AROONOSC_{aroon_length}'].round(2)
combined_data_5m['EMA_50'] = ta.ema(combined_data_5m['CLOSE'], length=ema_length).round(decimals)
combined_data_5m['STOCH-RSId'] = ta.stochrsi(
    combined_data_5m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSId_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_5m['STOCH-RSIk'] = ta.stochrsi(
    combined_data_5m['CLOSE'],
    length=stoch_rsi_length,
    smoothK=stoch_rsi_smoothK,
    smoothD=stoch_rsi_smoothD
)[f'STOCHRSIk_{stoch_rsi_length}_{stoch_rsi_length}_{stoch_rsi_smoothK}_{stoch_rsi_smoothD}'].round(2)
combined_data_5m['SUPERTREND'] = ta.supertrend(
    combined_data_5m['HIGH'],
    combined_data_5m['LOW'],
    combined_data_5m['CLOSE'],
    length=supertrend_length,
    multiplier=supertrend_mult
)[f'SUPERT_{supertrend_length}_{supertrend_mult}'].round(decimals)
combined_data_5m['Keltner-Upper'] = ta.kc(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCUe_{kc_length}_{kc_scalar}'].round(decimals)
combined_data_5m['Keltner-Basis'] = ta.kc(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCBe_{kc_length}_{kc_scalar}'].round(decimals)
combined_data_5m['Keltner-Lower'] = ta.kc(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=kc_length,
    scalar=kc_scalar
)[f'KCLe_{kc_length}_{kc_scalar}'].round(decimals)
combined_data_5m['ATR'] = ta.atr(
    high=combined_data_5m['HIGH'],
    low=combined_data_5m['LOW'],
    close=combined_data_5m['CLOSE'],
    length=atr_length
).round(decimals)


# Drop rows with NaN values (due to indicator calculation)
combined_data_5m.dropna(inplace=True)

# Display the data with indicators
print("5 min Data with indicators:")
combined_data_5m.head()

5 min Data with indicators:


,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,AROON-OSC,EMA_50,STOCH-RSId,STOCH-RSIk,SUPERTREND,Keltner-Upper,Keltner-Basis,Keltner-Lower,ATR
datetime,,,,,,,,,,,,,,,,
2005-01-03 05:55:00,102.75,102.76,102.71,102.74,0,0.0,0.0,-48.0,102.74,39.55,39.18,102.51,102.83,102.74,102.65,0.06
2005-01-03 06:00:00,102.75,102.79,102.73,102.76,0,0.0,0.0,-48.0,102.74,44.43,55.26,102.54,102.83,102.74,102.66,0.06
2005-01-03 06:05:00,102.75,102.79,102.75,102.79,0,0.0,0.0,-48.0,102.75,55.80,72.96,102.55,102.83,102.75,102.66,0.06
2005-01-03 06:10:00,102.78,102.79,102.74,102.78,0,0.0,0.0,-48.0,102.75,71.42,86.04,102.55,102.84,102.75,102.67,0.06
2005-01-03 06:15:00,102.79,102.79,102.75,102.77,0,0.0,0.0,-48.0,102.75,81.66,85.98,102.55,102.84,102.75,102.67,0.06


In [41]:
data_5m_2018 = combined_data_5m.loc['2018-01-01':'2018-12-31']

matching_rows = data_5m_2018[data_5m_2018['CLOSE'] == data_5m_2018['SUPERTREND']]
print(f'Number of matching rows: {len(matching_rows)}')
matching_rows.head()

Number of matching rows: 55


,OPEN,HIGH,LOW,CLOSE,TICKVOL,VOL,SPREAD,AROON-OSC,EMA_50,STOCH-RSId,STOCH-RSIk,SUPERTREND,Keltner-Upper,Keltner-Basis,Keltner-Lower,ATR
datetime,,,,,,,,,,,,,,,,
2018-01-02 18:50:00,112.234,112.280,112.220,112.27,0,0.0,0.0,-36.0,112.24,35.99,67.62,112.27,112.30,112.24,112.18,0.04
2018-01-09 23:35:00,112.319,112.341,112.304,112.34,0,0.0,0.0,-36.0,112.30,85.68,85.58,112.34,112.35,112.30,112.26,0.03
2018-01-11 03:10:00,111.729,111.734,111.707,111.73,0,0.0,0.0,42.0,111.79,0.19,0.58,111.73,111.85,111.79,111.73,0.04
2018-01-18 22:20:00,110.986,110.986,110.900,110.93,0,0.0,0.0,28.0,111.00,35.29,15.64,110.93,111.07,111.00,110.92,0.05
2018-01-23 09:55:00,110.418,110.427,110.331,110.36,0,0.0,0.0,-96.0,110.45,68.87,45.92,110.36,110.54,110.45,110.36,0.06


In [63]:
plotly_data = combined_data_5m.loc['2021-01-01':'2021-01-31']

# Create a Plotly figure with subplots
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, 
                    vertical_spacing=0.04, 
                    row_heights=[0.10, 0.75, 0.15, 0.15],
                    subplot_titles=(
                        'ATR 14',
                        'EMA 50 | Supertrend 50, 3.0 | Keltner Channels 20, 1.5',
                        'Aroon Oscillator 50',
                        'Stochastic RSI 14, 3, 3'
                    )
                )

# Add candlestick chart
fig.add_trace(go.Candlestick(x=plotly_data.index,
                             open=plotly_data['OPEN'],
                             high=plotly_data['HIGH'],
                             low=plotly_data['LOW'],
                             close=plotly_data['CLOSE'],
                             ),
            row=2, col=1)

# Add ATR to top of chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['ATR'],
    mode='lines',
    line=go.scatter.Line(color='yellow'),
    name='ATR'
), row=1, col=1)
# Add indicators to candlestick chart
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['EMA_50'],
    mode='lines',
    line=go.scatter.Line(color='red'),
    name='EMA 50'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['SUPERTREND'],
    mode='lines',
    line=go.scatter.Line(color='green'),
    name='SUPERTREND'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Upper'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Upper'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Basis'],
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Keltner-Basis'
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['Keltner-Lower'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Keltner-Lower'
), row=2, col=1)

# Add Aroon Oscillator
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['AROON-OSC'],
    mode='lines',
    line=go.scatter.Line(color='purple'),
    name='Aroon Oscillator'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[0] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='grey', dash='dash'),
    name='Aroon Osc Zero Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Mid Line'
), row=3, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[-60] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='blue', dash='dash'),
    name='Aroon Osc Upper Line'
), row=3, col=1)

# Add Stochastic RSI
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSId'],
    mode='lines',
    line=go.scatter.Line(color='orange'),
    name='Stochastic RSI %D'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=plotly_data['STOCH-RSIk'],
    mode='lines',
    line=go.scatter.Line(color='blue'),
    name='Stochastic RSI %K'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[20] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='red', dash='dash'),
    name='Oversold Threshold'
), row=4, col=1)
fig.add_trace(go.Scatter(
    x=plotly_data.index,
    y=[80] * len(plotly_data),
    mode='lines',
    line=go.scatter.Line(color='green', dash='dash'),
    name='Overbought Threshold'
), row=4, col=1)


fig.update_layout(
    title='EURUSD 5m Candlestick Chart with Indicators January 2021',
    xaxis2=dict(
        rangeslider=dict(visible=False),
        rangebreaks=[
            dict(bounds=["sat", "mon"]),  # hide weekends
            dict(values=["2021-01-08 17:00", "2021-01-10 17:00"])
        ]
    ),
    yaxis=dict(title='Price', autorange=True),
    dragmode='zoom',
    uirevision='dynamic',
)

fig.write_html('./plotly/test_plot_EURUSD_5m.html')

In [49]:
class SuperTrendStrategy(bt.Strategy):
    params = dict(
        risk_per_trade=0.0025,
        rr_ratio=2.8,
    )

    def __init__(self):
        self.buy_order = None
        self.sell_order = None
        self.trade_log = []
        self.open_trades = {}
        # self.data_1m = self.datas[0]
        # self.data_5m = self.datas[1]
        # Indicators
        self.supertrend = self.data.supertrend
        self.ema = self.data.ema_50
        self.aroon = self.data.aroon_osc
        self.stoch_rsi_d = self.data.stoch_rsi_d
        self.stoch_rsi_k = self.data.stoch_rsi_k
        self.kc_upper = self.data.keltner_upper
        self.kc_basis = self.data.keltner_basis
        self.kc_lower = self.data.keltner_lower
        self.atr = self.data.atr
        

    def log(self, *args, dt=None):
        '''Logging function for this strategy'''
        dt = dt or self.datas[0].datetime.date(0)
        print('%s, %s' % (dt.isoformat(), ', '.join(map(str, args))))
    
    def calculate_take_profit(self, current_price, stop_loss, is_buy):
            if is_buy:
                return current_price + ((current_price - stop_loss) * self.params.rr_ratio)
            else:
                return current_price - ((stop_loss - current_price) * self.params.rr_ratio)

    def calculate_volume(self, current_price, stop_loss):
        # Calculate position size based on risk
        account_balance = self.broker.getvalue()
        risk_amount = account_balance * self.params.risk_per_trade
        stop_loss_distance = abs(current_price - stop_loss)

        # Ensure stop loss distance is not zero or too small
        if stop_loss_distance == 0:
            self.log("Skipping trade due to stop loss distance being 0")
            return None

        # Calculate volume in lots (1 lot = 100,000 units)
        volume = risk_amount / stop_loss_distance / multiplier
        # if volume < 0:
        #     self.log(f"Skipping trade due to invalid volume - Volume: {volume}")
        #     return None
        # self.log(f"Calculated volume: {volume:.2f} lots")
        return round(volume, 2)           

    def next(self):
        if self.position:
            return
        
        # Fetch values
        current_price = self.data.close[0]
        current_high_range = self.data.high.get(size=5)
        current_low_range = self.data.low.get(size=5)
        recent_highs = self.data.high.get(size=10)
        recent_lows = self.data.low.get(size=10)

        if len(current_high_range) < 5 or len(current_low_range) < 5:
            self.log("Not enough data points to calculate current high and low.")
            return
        current_high = max(current_high_range)
        current_low = min(current_low_range)


        # Check if there are enough data points
        if len(recent_highs) < 10 or len(recent_lows) < 10:
            self.log("Not enough data points to calculate recent high and low.")
            return
        recent_high = max(recent_highs)
        recent_low = min(recent_lows)
        
        recent_high_index = self.data.high.get(size=10).index(recent_high)
        recent_low_index = self.data.low.get(size=10).index(recent_low)

        ago_recent_high = -(10 - recent_high_index)
        ago_recent_low = -(10 - recent_low_index)

        kc_upper_at_recent_high = self.kc_upper[ago_recent_high]
        kc_lower_at_recent_low = self.kc_lower[ago_recent_low]
        kc_basis = self.kc_basis[0]
        atr = self.atr[0]
        supertrend = self.supertrend[0]
        ema = self.ema[0]
        aroon = self.aroon[0]
        stoch_k = self.stoch_rsi_k[0]
        stoch_k_recent_max = max(self.stoch_rsi_k.get(size=5))
        stoch_k_recent_min = min(self.stoch_rsi_k.get(size=5))
        stoch_d = self.stoch_rsi_d[0]

        # Set Aroon Oscillator threshold
        aroon_threshold = 40

        # Log indicator values
        # print(f"Current Price: {current_price:.5f}, Supertrend: {supertrend:.5f}, EMA: {ema:.5f}, Aroon: {aroon:.2f}, Stoch K: {stoch_k:.2f}, Stoch D: {stoch_d:.2f}")

        # Calculate stop loss and take profit
        stop_loss = round(supertrend, decimals)
        take_profit_buy = self.calculate_take_profit(current_price, stop_loss, is_buy=True)
        take_profit_sell = self.calculate_take_profit(current_price, stop_loss, is_buy=False)
        volume = self.calculate_volume(current_price, stop_loss)

        if volume is None or volume <= 0:
            self.log("Volume calculation resulted in None or non-positive value.")
            return
        
        
        # Long entry
        if (
            not self.buy_order 
            and not self.sell_order
            and current_price > supertrend 
            and current_price > ema
            and recent_high > kc_upper_at_recent_high
            and current_low <= kc_basis 
            and aroon > aroon_threshold 
            and stoch_k_recent_min < 30 
            and stoch_d < stoch_k
        ):
            main_order = self.buy(size=abs(volume), exectype=bt.Order.Market, transmit=False)
            stop_order = self.sell(
                size=abs(volume), exectype=bt.Order.Stop,
                price=stop_loss, parent=main_order, transmit=False
            )
            profit_order = self.sell(
                size=abs(volume), exectype=bt.Order.Limit,
                price=take_profit_buy, parent=main_order, transmit=True
            )
            self.buy_order = main_order

        # Short entry
        elif (
            not self.sell_order
            and not self.buy_order
            and current_price < supertrend 
            and current_price < ema 
            and recent_low < kc_lower_at_recent_low
            and current_high >= kc_basis
            and aroon < -aroon_threshold
            and stoch_k_recent_max > 70 
            and stoch_d > stoch_k
        ):
            main_order = self.sell(size=abs(volume), exectype=bt.Order.Market, transmit=False)
            stop_order = self.buy(
                size=abs(volume), exectype=bt.Order.Stop,
                price=stop_loss, parent=main_order, transmit=False
            )
            profit_order = self.buy(
                size=abs(volume), exectype=bt.Order.Limit,
                price=take_profit_sell, parent=main_order, transmit=True
            )
            self.sell_order = main_order

    def log_trade(self, trade_info):
        """Log a trade into the trade log."""
        self.trade_log.append(trade_info)

    def notify_order(self, order):
        if order.status in [order.Submitted, order.Accepted]:
            return

        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f"BUY EXECUTED {order.ref} Price: \
{order.executed.price:.{decimals}f} \
Size: {order.executed.size:.2f}")
                self.buy_order = None  # Clear buy order
                self.open_trades[order.ref] = {
                    "TRADE NUMBER": len(self.trade_log) + 1,
                    "SYMBOL": "EURUSD",
                    "OPEN TIME": self.data.datetime.datetime(0),
                    "VOLUME": self.calculate_volume(self.data.close[0], self.supertrend[0]),
                    "SIDE": "BUY",
                    "OPEN PRICE": round(order.executed.price, decimals),
                    "STOP LOSS": round(self.supertrend[0], decimals),
                    "TAKE PROFIT": round(self.calculate_take_profit(self.data.close[0], self.supertrend[0], is_buy=True), decimals),
                }
                self.log_trade(self.open_trades[order.ref])
            elif order.issell():
                self.log(f"SELL EXECUTED {order.ref} Price: {order.executed.price:.{decimals}f} Size: {order.executed.size:.2f}")
                self.sell_order = None  # Clear sell order
                self.open_trades[order.ref] = {
                    "TRADE NUMBER": len(self.trade_log) + 1,
                    "SYMBOL": "EURUSD",
                    "OPEN TIME": self.data.datetime.datetime(0),
                    "VOLUME": self.calculate_volume(self.data.close[0], self.supertrend[0]),
                    "SIDE": "SELL",
                    "OPEN PRICE": round(order.executed.price, decimals),
                    "STOP LOSS": round(self.supertrend[0], decimals),
                    "TAKE PROFIT": round(self.calculate_take_profit(self.data.close[0], self.supertrend[0], is_buy=False), decimals),
                    "CLOSE TIME": None,
                    "CLOSE PRICE": None,
                    "PROFIT": None,
                    "RUNNING BALANCE": None,
                }
                self.log_trade(self.open_trades[order.ref])

            # Check if the order is closing an existing trade
            if order.exectype in [bt.Order.Stop, bt.Order.Limit] and order.parent:
                parent_ref = order.parent.ref
                if parent_ref in self.open_trades:
                    trade_info = self.open_trades.pop(parent_ref, {})
                    trade_info.update({
                        "CLOSE TIME": self.data.datetime.datetime(0),
                        "CLOSE PRICE": round(order.executed.price, decimals),
                        "PROFIT": round(order.executed.pnl, 2),
                        "RUNNING BALANCE": round(self.broker.getvalue(), 2),
                    })
                    self.log_trade(trade_info)
                    self.log(f"TRADE CLOSED {parent_ref} by {order.ref}, Gross PnL={order.executed.pnl:.2f}")

            self.bar_executed = len(self)

        elif order.status in [order.Canceled, order.Margin, order.Rejected]:
            self.log(f'Order Canceled/Margin/Rejected: {order.ref}, Status: {order.getstatusname()}')
            if order.status == order.Margin:
                self.log(f"Margin call: Not enough margin for order {order.ref}")
            elif order.status == order.Rejected:
                self.log(f"Order rejected: {order.ref}")

        self.order = None

    # def notify_trade(self, trade):
    #     if not trade.isclosed:
    #         return

    #     self.log(f"TRADE CLOSED trade.myref={final_ref}, Gross PnL={trade.pnl:.2f}")

    #     trade_info = self.open_trades.pop(final_ref, {})
        
    #     trade_info.update({
    #         "CLOSE TIME": self.data.datetime.datetime(0),
    #         "CLOSE PRICE": round(trade.price, 5),
    #         "PROFIT": round(trade.pnl, 2),
    #         "RUNNING BALANCE": round(self.broker.getvalue(), 2),
    #     })
    #     self.log_trade(trade_info)

    def stop(self):
        # Save trades to a DataFrame at the end of the backtest
        self.trade_df = pd.DataFrame(self.trade_log)

        # Filter out canceled orders
        self.trade_df = self.trade_df[self.trade_df['CLOSE PRICE'].notna()]

        # Remove duplicates
        self.trade_df = self.trade_df.drop_duplicates()

        self.trade_df.to_csv(
            f'./back_test_data/{symbol}_{year}_to_{end_year}_5m_ST{supertrend_length}_EMA{ema_length}_\
AROON{aroon_length}.csv',
            index=False
        )

In [50]:
# Convert the pandas DataFrame to a Backtrader data feed
class PandasData(bt.feeds.PandasData):
    lines = (
        'ema_50',
        'stoch_rsi_k',
        'stoch_rsi_d',
        'supertrend',
        'aroon_osc',
        'keltner_upper',
        'keltner_basis',
        'keltner_lower',
        'atr',
    )
    params = (
        ('datetime', None),
        ('open', 'OPEN'),
        ('high', 'HIGH'),
        ('low', 'LOW'),
        ('close', 'CLOSE'),
        ('volume', 'VOL'),
        ('ema_50', 'EMA_50'),
        ('stoch_rsi_k', 'STOCH-RSIk'),
        ('stoch_rsi_d', 'STOCH-RSId'),
        ('supertrend', 'SUPERTREND'),
        ('aroon_osc', 'AROON-OSC'),
        ('keltner_upper', 'Keltner-Upper'),
        ('keltner_basis', 'Keltner-Basis'),
        ('keltner_lower', 'Keltner-Lower'),
        ('atr', 'ATR'),
    )

test_data_1m = combined_data_1m.loc[f'{year}-01-01':f'{end_year}-12-31']
test_data_5m = combined_data_5m.loc[f'{year}-01-01':f'{end_year}-12-31']

data_feed = PandasData(dataname=test_data_5m)

comm_info = bt.CommInfoBase(
    commission=0.02,
    leverage=30,
    margin=1 / 30,
    mult=multiplier
)

# Set up the Backtrader environment
cerebro = bt.Cerebro()
cerebro.addstrategy(SuperTrendStrategy)
cerebro.adddata(data_feed)
# cerebro.resampledata(data_feed, timeframe=bt.TimeFrame.Minutes, compression=5, name='5m')
cerebro.broker.set_cash(2500)
cerebro.broker.addcommissioninfo(comm_info)

# Run the backtest
cerebro.run(style='candlestick')
print('Final Portfolio Value: %.2f' % cerebro.broker.getvalue())
# print('Number of trades: %d' % len())
# print('Max Drawdown: %.2f' % cerebro.broker.getdrawdown().max.drawdown)
# Plot the results
# cerebro.plot(show=True)
# Dates to plot
# from_date = '2018-12-21'
# to_date = '2018-12-31'
# fig = cerebro.plot(
#     style='candelstick',
#     volume=False,
#     fromdate=pd.Timestamp(from_date),
#     todate=pd.Timestamp(to_date)
# )[0][0]
# fig.savefig('backtrader_plot_5m_2018_aroon24_ema50_st50.pdf', format='pdf', dpi=300)


2008-01-01, Not enough data points to calculate current high and low.
2008-01-01, Not enough data points to calculate current high and low.
2008-01-01, Not enough data points to calculate current high and low.
2008-01-01, Not enough data points to calculate current high and low.
2008-01-01, Not enough data points to calculate recent high and low.
2008-01-01, Not enough data points to calculate recent high and low.
2008-01-01, Not enough data points to calculate recent high and low.
2008-01-01, Not enough data points to calculate recent high and low.
2008-01-01, Not enough data points to calculate recent high and low.
2008-01-02, SELL EXECUTED 21867 Price: 111.53 Size: -0.78
2008-01-02, BUY EXECUTED 21868 Price: 111.62 Size: 0.78
2008-01-02, TRADE CLOSED 21867 by 21868, Gross PnL=-7.02
2008-01-02, Order Canceled/Margin/Rejected: 21869, Status: Canceled
2008-01-02, SELL EXECUTED 21870 Price: 111.27 Size: -0.20
2008-01-02, BUY EXECUTED 21872 Price: 110.41 Size: 0.20
2008-01-02, TRADE CLOS